# EDA of the ME4OH dataset filtered to the North Atlantic Basin


In [19]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from Data.load_sst_ofam3_data import NA_DATA_PATH
from plotting_functions import plot_ohc_map, plot_sst_map, plot_compare_ff_insitu

In [ ]:
COUNTRIES_URL = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
WORLD = gpd.read_file(COUNTRIES_URL).rename({"ADMIN": "name"}, axis="columns")
regions = ['Southern Europe', 'Northern Africa', 'Western Africa', 'Western Europe', 'Northern Europe', 'Central America', 'South America', 'Northern America', 'Caribbean']
NA = WORLD[WORLD['SUBREGION'].isin(regions)]

## EDA - OHC

In [ ]:
df = pd.read_csv("Data/ME4OH_EN411_OFAM3/NA_1993_2014.csv")

# Convert dates to datetime, save year month and year indication columns (aids plotting)
df['Date'] = pd.to_datetime(df['Date'])
df['year_month'] = df['Date'].dt.to_period('M')
df['year'] = df['Date'].dt.to_period('Y')
df

### Plot OHC in latitude and longitude

In [ ]:
month_list = pd.date_range('1993-1-1','2014-12-31', freq='MS').strftime('%Y-%m').tolist()
num_plots = 10
plot_months = [month_list[i] for i in range(0, len(month_list), len(month_list)//num_plots)]

In [ ]:
for n,g in df.groupby('year_month'):
    if str(n) in plot_months:
        plot_ohc_map(g, n, min(df.OHC), max(df.OHC), area="NA")

In [ ]:
for n,g in df.groupby('year'):
    plot_ohc_map(g, n, min(df.OHC), max(df.OHC), area="NA")

### Plot by platform (OHC dataset)

In [ ]:
# Filter by platforms
platforms = {
    "Argo": df.loc[df['Project name'].str.contains('ARGO')].sort_values('Date').reset_index(drop=True),
    "WOD": df.loc[df['Project name'].str.contains('WOD')].sort_values('Date').reset_index(drop=True),
    "GTSPP": df.loc[df['Project name'].str.contains('GTSPP')].sort_values('Date').reset_index(drop=True),
    "ABSO":  df.loc[df['Project name'].str.contains('ASBO')].sort_values('Date').reset_index(drop=True),
}

In [ ]:
colours = [ "#631ff3", "#ffb000", "#fe5100", "#1589e8", "#dc267f"]  # IBM colourblind palette
year_list = pd.date_range('1993-1-1','2014-12-31', freq='YS').strftime('%Y').tolist()

In [ ]:
for year in year_list:
    _, ax = plt.subplots(figsize=(12,6))
    NA.plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.75)  # Add countries
    for i, (name, data) in enumerate(platforms.items()):
        year_data = data.loc[data["year"] == year]
        ax.scatter(year_data.Longitude, year_data.Latitude, s=1, label=name, c=colours[i])
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_facecolor("lightblue")
    ax.set_xlim([-80, 0])
    ax.set_ylim([0, 60])
    plt.legend(bbox_to_anchor=(1,1))
    plt.title(year)
    plt.savefig(f"images/platforms_NA_by_year/platforms_NA_{year}.png")
    plt.show()
    plt.close()

## EDA - SST

Note: Assumptions as of 30/06/26
- OFAM3 data is stored for Thursdays across the 1979-2014 time period.
- Pending confirmation from the ME4OH team, I have assumed that the data is the full-field SST values generated by the simulator model (OFAM3) averaged across a calendar week

In [ ]:
# ensure datetime and ISO week/year columns exist
na_df = pd.read_csv("Data/ME4OH_EN411_OFAM3/SST_NA_1993_2014.csv")
na_df['Date'] = pd.to_datetime(na_df['Date'])
na_df['iso_year'] = na_df['Date'].dt.isocalendar().year
na_df['iso_week'] = na_df['Date'].dt.isocalendar().week
na_df

In [ ]:
ref_date = pd.Timestamp("2014-06-19")
ref_year, ref_week = ref_date.isocalendar()[:2]

# keep only the same ISO week/year
week_df = na_df[
    (na_df['iso_year'] == ref_year) &
    (na_df['iso_week'] == ref_week)
].copy()

In [ ]:
plot_sst_map(week_df, f"Data from week containing: {ref_date.date()}", min(week_df.SST), max(week_df.SST), area="NA", background=True)

In [ ]:
date = '2014-06-19'
df = pd.read_csv(f"{NA_DATA_PATH}{date}.csv")

In [ ]:
plot_compare_ff_insitu(df, week_df, area="NA", title=f"Data from week containing: {ref_date.date()}")

In [9]:
# How many sensors per year?
na_df

,Date,Latitude,Longitude,SST,WMO Instrument Type,Project name,Platform number,iso_year,iso_week
0,1993-01-01,8.45,-29.649994,27.259743,401,WOD09XBTPA 123,14302123,1992,53
1,1993-01-01,7.35,-30.450012,27.571947,401,WOD09XBTPA 123,14302123,1992,53
2,1993-01-01,16.75,-56.950012,27.412489,401,WOD09XBTFR 4200,11408200,1992,53
3,1993-01-01,35.45,-12.149994,18.474380,401,WOD09XBT99,0,1992,53
4,1993-01-01,41.15,-65.450012,21.003906,401,WOD09XBTUS 3683,14095183,1992,53
...,...,...,...,...,...,...,...,...,...
1216523,2014-12-31,46.45,-14.950012,12.524034,831,ARGOR,6900974,2015,1
1216524,2014-12-31,0.05,-34.950012,27.388988,820,GTSPPTR 99,31001,2015,1
1216525,2014-12-31,15.05,-37.950012,25.955534,820,GTSPPTR 99,13008,2015,1
1216526,2014-12-31,38.85,-24.850006,16.508835,831,ARGOR,6902553,2015,1


In [10]:
na_df.groupby('iso_year').size()

iso_year
1992      214
1993    41670
1994    33076
1995    42334
1996    39036
1997    46730
1998    54146
1999    53210
2000    49112
2001    62992
2002    43432
2003    43704
2004    52490
2005    54860
2006    59782
2007    63010
2008    67080
2009    68670
2010    64688
2011    66640
2012    67326
2013    76140
2014    65784
2015      402
dtype: int64

In [18]:
na_df.groupby(['iso_year', 'iso_week']).size()

iso_year  iso_week
1992      53           214
1993      1            504
          2            798
          3            550
          4            410
                      ... 
2014      49          1602
          50          1638
          51          1150
          52           936
2015      1            402
Length: 1149, dtype: int64

In [16]:
np.mean(na_df.groupby(['iso_year', 'iso_week']).size())

np.float64(1058.7711053089643)